# Mount file เพื่อนำไฟล์มาใช้ในโปรแกรม

In [10]:

from google.colab import drive
drive.mount('/content/drive')
!ls /content/drive/MyDrive/aiforeveryone

Mounted at /content/drive
accident.mp4  cnn	       mri_scan.jpg  number8.png  traffic.png
ball.jpg      emails_spam.csv  number8n.png  traffic.mp4  ืีnumber8n.png


# โค้ด  9.1. Fine-tuning NER

In [ ]:
!pip install transformers datasets seqeval pythainlp


In [ ]:
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import DataCollatorForTokenClassification
from transformers import TrainingArguments, Trainer
import numpy as np
from seqeval.metrics import classification_report

# 1) เตรียมข้อมูลตัวอย่าง
texts = ["สมชาย ทำงานที่ กระทรวงศึกษาธิการ ประเทศไทย"]
tokens = [
    ["สม", "ชาย", "ทำ", "งาน", "ที่", "กระ", "ทร", "วง", "ศึก", "ษา", "ธิ", "การ", "ประเทศไทย"]
]
labels = [
    ["B-PER","I-PER","O","O","O","B-ORG","I-ORG","I-ORG","I-ORG","I-ORG","I-ORG","I-ORG","B-LOC"]
]

# สร้าง label2id, id2label
label_list = list(set([l for seq in labels for l in seq]))
label2id = {l:i for i,l in enumerate(label_list)}
id2label = {i:l for l,i in label2id.items()}

dataset = Dataset.from_dict({"tokens": tokens, "labels": labels})
dataset = DatasetDict({"train": dataset, "test": dataset})

# 2) โหลด tokenizer + model
model_name = "airesearch/wangchanberta-base-att-spm-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 3) Tokenize + align labels
def tokenize_and_align(batch):
    tokenized = tokenizer(batch["tokens"], is_split_into_words=True, truncation=True)
    new_labels = []
    for i, lbls in enumerate(batch["labels"]):
        word_ids = tokenized.word_ids(batch_index=i)
        aligned_labels = []
        prev_word = None
        for wid in word_ids:
            if wid is None:
                aligned_labels.append(-100)
            else:
                aligned_labels.append(label2id[lbls[wid]])
        new_labels.append(aligned_labels)
    tokenized["labels"] = new_labels
    return tokenized

tokenized_dataset = dataset.map(tokenize_and_align, batched=True)

# 4) เตรียมข้อมูลสำหรับ Training
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

# 5) Training arguments
training_args = TrainingArguments(
    output_dir="./thai-ner-bert",
    learning_rate=5e-5,
    num_train_epochs=5,
    per_device_train_batch_size=4,
    weight_decay=0.01,
    report_to="none" # Added to disable wandb logging
)

# 6) Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    tokenizer=tokenizer
)

trainer.train()

# 7) ทดสอบ
test_text = "อานันท์ เดินทางไปยัง กรุงเทพมหานคร"
tokens_test = ["อานันท์","เดินทาง","ไปยัง","กรุง","เทพ","มหา","นคร"]

encoding = tokenizer(tokens_test, is_split_into_words=True, return_tensors="pt")
outputs = model(**encoding)
pred_ids = np.argmax(outputs.logits.detach().numpy(), axis=2)[0]

print("ผลลัพธ์ NER:")
for tok, pid in zip(tokens_test, pred_ids):
    print(tok, "→", id2label[pid])

#โค้ด 9.2 หาคำที่มีความหมายใกล้เคียง

In [ ]:
from sentence_transformers import SentenceTransformer

# Load a pretrained embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Words/phrases related to BERT
words = ["BERT", "transformer", "Google AI", "bidirectional",
         "masked language model", "fine-tuning", "NLP"]

# Get embeddings (vectors)
embeddings = model.encode(words)

# Print vector for "BERT"
print("Vector for 'BERT':")
print(embeddings[0])  # This will be a long list of floats

# Show cosine similarity between BERT and other words
from sentence_transformers.util import cos_sim
for i in range(1, len(words)):
    score = cos_sim(embeddings[0], embeddings[i])
    print(f"BERT ↔ {words[i]} = {score.item():.3f}")

# โค้ด 9.3 หาคำเติมลงในช่องว่าง

In [ ]:
from transformers import pipeline

# Load a BERT-based fill-mask model (good for correcting words in context)
unmasker = pipeline("fill-mask", model="bert-base-uncased")

sentence = "He go to school yesterday."
result = unmasker("He [MASK] to school yesterday.")

print("Suggestions:", [r['token_str'] for r in result])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Device set to use cpu


Suggestions: ['went', 'came', 'returned', 'goes', 'got']


#โค้ด 9.4 Email spam

In [14]:
pip install transformers datasets accelerate pandas scikit-learn evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00


In [17]:
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
import numpy as np
import evaluate

# 1) โหลดข้อมูลจาก CSV
df = pd.read_csv("/content/drive/MyDrive/aiforeveryone/emails_spam.csv")

# สมมติ label: 1 = spam, 0 = ham
# แปลง 'ham' เป็น 0 และ 'spam' เป็น 1
df['label'] = df['label'].apply(lambda x: 1 if x == 'spam' else 0)

# แบ่ง train / test
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds  = Dataset.from_pandas(test_df.reset_index(drop=True))

datasets = DatasetDict({
    "train": train_ds,
    "test": test_ds
})

# 2) เลือกโมเดล BERT
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 3) ฟังก์ชัน tokenize ข้อความ
max_length = 128

def preprocess_function(examples):
    # Combine 'subject' and 'body' to create the text input
    # Handle potential NaN in 'body' by converting to empty string
    texts = [f"{s} {b if pd.notna(b) else ''}" for s, b in zip(examples["subject"], examples["body"])]
    return tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=max_length
    )

tokenized_datasets = datasets.map(preprocess_function, batched=True)

# 4) เตรียมโมเดล BERT สำหรับ classification 2 class
num_labels = 2

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

# 5) Metric สำหรับประเมินผล
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="macro")["f1"]
    }

# 6) Training arguments
training_args = TrainingArguments(
    output_dir="./bert-spam-output",
    eval_strategy="epoch", # Changed from evaluation_strategy
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none" # Disable wandb logging
)

# Map labels to integer IDs for the model
def convert_labels_to_ids(example):
    return {'labels': example['label']}

train_dataset = tokenized_datasets["train"].map(convert_labels_to_ids)
eval_dataset  = tokenized_datasets["test"].map(convert_labels_to_ids)

# 7) สร้าง Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# 8) Train
trainer.train()

# 9) Evaluate บน test set
metrics = trainer.evaluate()
print(metrics)

# 10) Save โมเดลไว้ใช้ภายหลัง
trainer.save_model("./bert-spam-model")
tokenizer.save_pretrained("./bert-spam-model")


Map:   0%|          | 0/24 [00:00<?, ? examples/s]

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/24 [00:00<?, ? examples/s]

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

/tmp/ipython-input-329048775.py:96: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.664010,0.500000,0.333333
2,No log,0.639504,0.833333,0.828571
3,No log,0.625549,0.833333,0.828571


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.6395041346549988, 'eval_accuracy': 0.8333333333333334, 'eval_f1': 0.8285714285714285, 'eval_runtime': 2.3315, 'eval_samples_per_second': 2.573, 'eval_steps_per_second': 0.429, 'epoch': 3.0}


('./bert-spam-model/tokenizer_config.json',
 './bert-spam-model/special_tokens_map.json',
 './bert-spam-model/vocab.txt',
 './bert-spam-model/added_tokens.json',
 './bert-spam-model/tokenizer.json')

ทำนาย

In [19]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_path = "./bert-spam-model"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

id2label = {0: "NOT_SPAM", 1: "SPAM"}

def predict_email(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        pred = torch.argmax(probs, dim=-1).item()
        return {
            "label": id2label[pred],
            "probs": probs[0].tolist()
        }

# ทดลอง
email1 = "Congratulations! You have won a free gift card. Click here to claim now."
email2 = "Hi John, here is the meeting agenda for tomorrow."

print(predict_email(email1))
print(predict_email(email2))


{'label': 'SPAM', 'probs': [0.3773012161254883, 0.6226987838745117]}
{'label': 'SPAM', 'probs': [0.37518662214279175, 0.6248133778572083]}


#โค้ด 9.5 หาว่าประโยคขัดแย้งกันหรือไม่

In [ ]:
pip install transformers torch sentencepiece

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# 1. เลือกโมเดล NLI
#   - "joeddav/xlm-roberta-large-xnli" รองรับหลายภาษา (รวมถึงไทยค่อนข้างดี)
#   - หรือจะเปลี่ยนเป็นโมเดล NLI อื่นก็ได้ภายหลัง
MODEL_NAME = "joeddav/xlm-roberta-large-xnli"

# 2. โหลด tokenizer และ model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

# 3. สร้าง NLI pipeline
nli = pipeline("text-classification", model=model, tokenizer=tokenizer)

def nli_predict(premise: str, hypothesis: str):
    """
    ทำ NLI ระหว่างประโยค A (premise) และ B (hypothesis)
    คืนค่า label ('ENTAILMENT', 'CONTRADICTION', 'NEUTRAL') และ score
    """
    # โมเดล XNLI คาดว่า input จะอยู่ในรูป (premise, hypothesis)
    input_text = f"{premise} </s> {hypothesis}"

    result = nli(input_text, truncation=True)[0]
    label = result["label"].upper()
    score = float(result["score"])

    # mapping label ให้เข้าใจง่าย (บางโมเดลใช้ชื่อ slightly ต่างกัน)
    if "CONTRADICTION" in label:
        label_th = "ขัดแย้ง (contradiction)"
    elif "ENTAILMENT" in label:
        label_th = "สนับสนุน (entailment)"
    else:
        label_th = "เป็นกลาง (neutral)"

    return {
        "premise": premise,
        "hypothesis": hypothesis,
        "raw_label": label,
        "label_th": label_th,
        "score": score,
    }

if __name__ == "__main__":
    # ตัวอย่างภาษาไทยตามที่ถาม
    A = "สัตว์ป่าอยู่ในป่า"
    B = "สัตว์ป่าหากินในเมืองใหญ่"

    result = nli_predict(A, B)

    print("Premise (ประโยค A):", result["premise"])
    print("Hypothesis (ประโยค B):", result["hypothesis"])
    print("ผลอนุมานเชิงภาษา:")
    print("  Label (ดิบ):", result["raw_label"])
    print("  ความหมาย:", result["label_th"])
    print("  ความมั่นใจ (score):", result["score"])


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/734 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


Premise (ประโยค A): สัตว์ป่าอยู่ในป่า
Hypothesis (ประโยค B): สัตว์ป่าหากินในเมืองใหญ่
ผลอนุมานเชิงภาษา:
  Label (ดิบ): CONTRADICTION
  ความหมาย: ขัดแย้ง (contradiction)
  ความมั่นใจ (score): 0.9957218170166016


In [ ]:
tests = [
    ("เด็กผู้ชายกำลังเตะบอล", "มีคนกำลังเล่นกีฬา"),
    ("ฝนกำลังตกหนัก", "อากาศแจ่มใสไม่มีเมฆ"),
    ("ผู้หญิงคนหนึ่งกำลังอ่านหนังสือ", "เธอกำลังทำงานบ้าน"),
]

for A, B in tests:
    r = nli_predict(A, B)
    print("=" * 60)
    print("A:", r["premise"])
    print("B:", r["hypothesis"])
    print("=>", r["label_th"], "| score =", r["score"])
